# Light OCR on weak-Tesseract pages

Runs **PP-OCRv5** and **Surya** on the 297 pages where Tesseract was weak (mean confidence < 60 or 0 words), and scores them against Tesseract and KDL (the ceiling) with OHR-Bench `evidence_context` and answers.

Run order matters: PaddlePaddle and torch pin different NVIDIA libraries, so **Paddle runs first**, its results are written to disk, and installing Surya afterwards puts torch's libraries back.

1. Runtime: **L4 GPU**.
2. Build the upload locally: `python -m ondemand.ocr_bundle`.
3. Run the cells top to bottom. Every loop resumes, so re-running a cell after a crash only redoes the missing pages.
4. Download `light_ocr_results.zip` at the end.

In [ ]:
import json
import os
import subprocess
import time
import zipfile

os.chdir("/content")
if not os.path.exists("light_ocr/pages.jsonl"):
    from google.colab import files
    uploaded = files.upload()
    zipfile.ZipFile(next(iter(uploaded))).extractall("light_ocr")
pages = [json.loads(line) for line in open("light_ocr/pages.jsonl")]
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()

def done_units(name):
    path = f"light_ocr/light_ocr_{name}.jsonl"
    return {json.loads(l)["unit"] for l in open(path)} if os.path.exists(path) else set()

def run_all(name, transcribe, meta):
    done = done_units(name)
    todo = [p for p in pages if p["unit"] not in done]
    print(f"{name}: {len(done)} done, {len(todo)} to go")
    with open(f"light_ocr/light_ocr_{name}.jsonl", "a") as f:
        for i, p in enumerate(todo, 1):
            started = time.perf_counter()
            try:
                text, error = transcribe(p), None
            except Exception as exc:
                text, error = "", f"{type(exc).__name__}: {exc}"[:200]
                print("fail", p["unit"], error)
            f.write(json.dumps({"unit": p["unit"], "text": text, "seconds": time.perf_counter() - started, "error": error}, ensure_ascii=False) + "\n")
            f.flush()
            if i % 50 == 0 or i == len(todo):
                print(f"  {i}/{len(todo)}")
    json.dump({**meta, "gpu": gpu}, open(f"light_ocr/meta_{name}.json", "w"))

print(len(pages), "pages,", sum(bool(p["evidence"]) for p in pages), "with evidence, gpu:", gpu)

In [ ]:
!pip install -q pymupdf
import fitz

MAX_SIDE = 4000
os.makedirs("light_ocr/png", exist_ok=True)
for p in pages:
    p["png"] = "light_ocr/png/" + os.path.basename(p["file"])[:-4] + ".png"
    if not os.path.exists(p["png"]):
        with fitz.open("light_ocr/" + p["file"]) as doc:
            page = doc[0]
            dpi = 200
            side = max(page.rect.width, page.rect.height) * dpi / 72
            if side > MAX_SIDE:
                dpi = int(dpi * MAX_SIDE / side)
            page.get_pixmap(dpi=dpi).save(p["png"])
print("rendered", len(pages), f"pages at 200 dpi (capped at {MAX_SIDE} px on the long side)")

## PP-OCRv5 (server det + rec)

The last line restores the NVIDIA libraries that torch pins, so the Surya cells still work in this runtime.

In [ ]:
!pip install -q paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q "paddleocr>=3.0,<4"
!pip install -q nvidia-nccl-cu12==2.28.9 nvidia-cudnn-cu12==9.19.0.56 nvidia-cusparselt-cu12==0.7.1

In [ ]:
from paddleocr import PaddleOCR

started = time.perf_counter()
ocr = PaddleOCR(text_detection_model_name="PP-OCRv5_server_det", text_recognition_model_name="PP-OCRv5_server_rec",
                use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False, device="gpu:0")
ocr.predict(pages[0]["png"])
load = time.perf_counter() - started

def paddle_text(p):
    return "\n".join(t for r in ocr.predict(p["png"]) for t in r["rec_texts"])

run_all("ppocrv5", paddle_text, {"model": "PP-OCRv5_server_det + PP-OCRv5_server_rec", "load_seconds": load})

## EasyOCR (candidate B)

Surya 0.22 spawns a vLLM server through Docker, which Colab does not have, so the second candidate is EasyOCR: a detector plus a CRNN recogniser, about 100M parameters, pure torch, no server. Languages `en` + `fr`, which share the Latin recogniser.

In [ ]:
!pip install -q easyocr

In [ ]:
import importlib.metadata

import easyocr
import torch

version = importlib.metadata.version("easyocr")
started = time.perf_counter()
reader = easyocr.Reader(["en", "fr"], gpu=torch.cuda.is_available())
reader.readtext(pages[0]["png"], detail=0)
load = time.perf_counter() - started
params = sum(x.numel() for m in (reader.detector, reader.recognizer) for x in m.parameters())
print(f"easyocr {version}: {params / 1e6:.0f}M parameters (KDL nano is 1200M)")

def easyocr_text(p):
    return "\n".join(reader.readtext(p["png"], detail=0))

run_all("easyocr", easyocr_text, {"model": f"easyocr {version} en+fr", "parameters": params, "load_seconds": load})

## Surya 0.16.7 (optional, candidate C)

The last Surya release that runs in-process on torch, before the Docker/vLLM backend. It is optional: if the install or the import fails, the cell prints why and the rest of the notebook still works. Run it last, since it pins older torch-side packages.

In [ ]:
!pip install -q "surya-ocr==0.16.7"

import inspect

import torch

def torch_params(*roots, depth=3):
    found, stack, seen = {}, [(r, 0) for r in roots], set()
    while stack:
        obj, level = stack.pop()
        for value in list(getattr(obj, "__dict__", {}).values()):
            if isinstance(value, torch.nn.Module):
                found[id(value)] = value
            elif level < depth and hasattr(value, "__dict__") and id(value) not in seen:
                seen.add(id(value))
                stack.append((value, level + 1))
    return sum(x.numel() for m in found.values() for x in m.parameters())

try:
    from PIL import Image
    from surya.detection import DetectionPredictor
    from surya.foundation import FoundationPredictor
    from surya.recognition import RecognitionPredictor

    started = time.perf_counter()
    recognition = RecognitionPredictor(FoundationPredictor())
    detection = DetectionPredictor()
    accepted = inspect.signature(recognition.__call__).parameters
    print("surya signature", inspect.signature(recognition.__call__))

    def surya_call(images):
        kwargs = {}
        for name, value in (("langs", [None] * len(images)), ("task_names", ["ocr_with_boxes"] * len(images)),
                            ("det_predictor", detection), ("detection_predictor", detection)):
            if name in accepted:
                kwargs[name] = value
        return recognition(images, **kwargs)

    surya_call([Image.open(pages[0]["png"]).convert("RGB")])
    load = time.perf_counter() - started
    params = torch_params(recognition, detection)
    print(f"surya 0.16.7: {params / 1e9:.2f}B parameters (KDL nano is 1.2B)")

    def surya_text(p):
        result = surya_call([Image.open(p["png"]).convert("RGB")])[0]
        return "\n".join(line.text for line in result.text_lines)

    run_all("surya", surya_text, {"model": "surya-ocr 0.16.7", "parameters": params, "load_seconds": load})
except Exception as exc:
    print("skipping surya:", type(exc).__name__, exc)

## Quality and timing

On the OHR-Bench flagged pages: `evidence_word_recall` = share of the gold evidence words present in the OCR text; `answer_found` = a gold answer string appears in the OCR text. KDL is the ceiling, Tesseract the floor. This cell runs with whichever OCR files exist.

## Line boxes and confidences (for the debugging portal)

Re-runs both engines keeping, per detected line, its polygon and recognition confidence. Coordinates are in the pixel space of the PNGs rendered above (200 dpi, long side capped at 4000), which is what the portal re-renders. About 7 minutes for both engines.

In [ ]:
def write_boxes(name, lines_of):
    path = f"light_ocr/light_ocr_boxes_{name}.jsonl"
    done = {json.loads(l)["unit"] for l in open(path)} if os.path.exists(path) else set()
    todo = [p for p in pages if p["unit"] not in done]
    print(f"{name} boxes: {len(done)} done, {len(todo)} to go")
    with open(path, "a") as f:
        for i, p in enumerate(todo, 1):
            try:
                lines, error = lines_of(p), None
            except Exception as exc:
                lines, error = [], f"{type(exc).__name__}: {exc}"[:200]
            f.write(json.dumps({"unit": p["unit"], "lines": lines, "error": error}, ensure_ascii=False) + "\n")
            f.flush()
            if i % 50 == 0 or i == len(todo):
                print(f"  {i}/{len(todo)}")

def paddle_lines(p):
    out = []
    for r in ocr.predict(p["png"]):
        polys = r.get("rec_polys", r.get("dt_polys", []))
        for text, score, poly in zip(r["rec_texts"], r["rec_scores"], polys):
            out.append({"text": text, "score": float(score), "poly": [[float(x), float(y)] for x, y in poly]})
    return out

def easyocr_lines(p):
    return [{"text": text, "score": float(score), "poly": [[float(x), float(y)] for x, y in poly]}
            for poly, text, score in reader.readtext(p["png"], detail=1)]

for name, engine, lines_of in (("ppocrv5", "ocr", paddle_lines), ("easyocr", "reader", easyocr_lines)):
    if engine in globals():
        write_boxes(name, lines_of)
    else:
        print(f"skipping {name} boxes: run its engine cell first in this runtime")

In [ ]:
import re
import statistics
import unicodedata
from collections import defaultdict

CANDIDATES = ("ppocrv5", "easyocr", "surya")

def as_text(x):
    if isinstance(x, (list, tuple)):
        return " ".join(as_text(i) for i in x)
    return "" if x is None else str(x)

def tokens(s):
    return re.findall(r"\w+", unicodedata.normalize("NFKC", as_text(s)).casefold())

def norm(s):
    return " ".join(tokens(s))

systems = {"tesseract": {p["unit"]: p["tesseract_text"] for p in pages}, "kdl": {p["unit"]: p["kdl_text"] for p in pages}}
timing = {}
for name in CANDIDATES:
    path = f"light_ocr/light_ocr_{name}.jsonl"
    if not os.path.exists(path):
        continue
    rows = [json.loads(l) for l in open(path)]
    systems[name] = {r["unit"]: r["text"] for r in rows}
    seconds = [r["seconds"] for r in rows]
    timing[name] = {"pages": len(rows), "failed": sum(bool(r.get("error")) for r in rows),
                    "empty": sum(not r["text"].strip() for r in rows), "total_seconds": round(sum(seconds), 1),
                    "seconds_per_page": round(statistics.mean(seconds), 3),
                    **json.load(open(f"light_ocr/meta_{name}.json"))}

pairs = [(p["unit"], e) for p in pages for e in p["evidence"]]
quality = {}
for name, text in systems.items():
    by = defaultdict(lambda: {"recall": [], "found": []})
    for unit, e in pairs:
        page_text = text.get(unit, "")
        have, want = set(tokens(page_text)), set(tokens(e["evidence_context"]))
        flat = norm(page_text)
        answers = e["answers"] if isinstance(e["answers"], (list, tuple)) else [e["answers"]]
        for group in (e["group"], "all"):
            by[group]["recall"].append(len(want & have) / max(len(want), 1))
            by[group]["found"].append(any(norm(a) and norm(a) in flat for a in answers))
    quality[name] = {g: {"n": len(v["found"]), "evidence_word_recall": round(100 * statistics.mean(v["recall"]), 1),
                         "answer_found": round(100 * statistics.mean(v["found"]), 1)} for g, v in sorted(by.items())}

json.dump({"quality": quality, "timing": timing}, open("light_ocr/quality.json", "w"), indent=1)
for name, groups in quality.items():
    for group, v in groups.items():
        print(f"{name:10s} {group:22s} n={v['n']:3d} evidence_word_recall {v['evidence_word_recall']:5.1f}  answer_found {v['answer_found']:5.1f}")
print(json.dumps(timing, indent=1))

In [ ]:
with zipfile.ZipFile("light_ocr_results.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for name in sorted(os.listdir("light_ocr")):
        if name.endswith((".jsonl", ".json")) and name != "pages.jsonl":
            zf.write(f"light_ocr/{name}", name)
from google.colab import files
files.download("light_ocr_results.zip")